## 1、准备数据

In [1]:
# 字典key是租户tenant_id，字典value是租户对应的user信息，也是字典。user字典的key是user_id
USER_DATABASE = {
    "school_a": {
        "u001": {"name": "虎哥", "course": "LangChain 开发指南"},
        "u002": {"name": "小明", "course": "Java开发入门"},
    },
    "school_b": {
        "u001": {"name": "涛哥", "course": "Python 入门"},
        "u002": {"name": "阿伟", "course": "VibeCoding 实战"},
    }
}

## 2、自定义Context Schema

In [2]:
from dataclasses import dataclass


@dataclass(frozen=True)
class UserContext:
    """Agent运行时上下文"""
    user_id: str
    tenant_id: str


## 3、在Tool中使用Context

In [3]:
from langchain.tools import tool, ToolRuntime


@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
    """查询当前登录用户的资料。"""

    # 获取context
    ctx = runtime.context

    # 根据user_id和tenant_id获取用户信息
    profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)

    return (
        f"姓名：{profile['name']}，"
        f"课程：{profile['course']}，"
    )

## 4、给Agent添加Context

In [5]:
from langchain.agents import create_agent

from dotenv import load_dotenv

load_dotenv()

# 创建Agent时指定context_schema
agent = create_agent(
    model="deepseek-flash",
    tools=[get_current_user_profile],
    context_schema=UserContext,  # 指定Context类型
    system_prompt="你是课程平台助手。需要用户资料时调用工具。"
)

from langchain.messages import HumanMessage

# 调用时通过Context传递用户信息
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询我正在学习的课程")]},
    context=UserContext(user_id="u001", tenant_id="school_a")
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hello, 帮我查询我正在学习的课程
================================== Ai Message ==================================

I'll look up your profile to check your course info.
Tool Calls:
  get_current_user_profile (call_00_AkgVag7J4ysoAcZYDv4O4584)
 Call ID: call_00_AkgVag7J4ysoAcZYDv4O4584
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥，课程：LangChain 开发指南，
================================== Ai Message ==================================

虎哥，你好！查询到你的资料如下：

- **姓名**：虎哥
- **正在学习的课程**：LangChain 开发指南

目前资料里只显示这一门课程。如果你还想查看学习进度、课程目录或其他课程信息，告诉我具体想了解什么，我再帮你看看～
